# ☁️ HELIX — Phase 6: Cloud Deployment

**MSc Data Science Project — Queen Mary University of London**
**Student:** Sawan Masih Nayak (250885248)
**Submission Year:** 2026

---

## Phase Overview

Phase 6 takes every artefact the previous five phases produced — four trained models, their scalers and encoders, and the 217-triple OWL/RDF knowledge graph — and turns them into a live, publicly reachable platform. The deployment consists of **two independent microservices on Google Cloud Run** (region `europe-west2`, project `helix-deploy-2026`):

- a **FastAPI backend** that loads all four models and the ontology at startup and serves predictions and SPARQL queries over a REST API, and
- a **Streamlit frontend** that consumes that API over HTTP and gives users a browser interface.

This notebook plays a different role from Phases 1–5. The deployment itself was executed from a **local PowerShell session** using the `gcloud` CLI — Cloud Run deployments cannot be driven from inside Colab — so Sections 1–3 *document* the method, configuration, and the build failures that shaped it, while Section 4 contains **executable verification cells** that test the live deployment end-to-end from this notebook. Running Section 4 constitutes the reproducible evidence that the platform is up and serving correct responses.

## Relationship to Research Questions

| Research Question | Phase 6 Contribution |
|---|---|
| RQ6 — System Integration | The whole phase: all analytical, predictive, and knowledge components served live through one platform |

## Module Coverage

| Module | Phase 6 Coverage |
|---|---|
| Cloud Computing | **Full** — containerisation, serverless deployment, resource configuration, scale-to-zero |
| Distributed Systems | Applied — two loosely-coupled services communicating over HTTP/REST |


---

# Section 1: Deployment Method

## 1.1 Why Cloud Native Buildpacks (and not a Dockerfile)

Both services are containerised **from source** using Cloud Native Buildpacks: `gcloud run deploy` inspects the source directory, detects a Python application, and assembles the container image itself. This was a deliberate choice over a hand-written Dockerfile:

1. **The build environment is declared, not scripted.** The Python version is pinned in a `.python-version` file and dependencies in `requirements.txt`; the buildpack guarantees the image matches them.
2. **Less to get wrong.** A Dockerfile is another program that can contain bugs; the buildpack path removes an entire class of them.
3. **Reproducibility.** Anyone with the repository and the two commands below can produce the same deployment.

## 1.2 The deployment commands

Executed from the repository root in PowerShell (documented here verbatim for reproducibility):

```powershell
# Backend — FastAPI service serving the four models + ontology
gcloud run deploy helix-backend `
  --source backend/ `
  --region europe-west2 `
  --allow-unauthenticated `
  --memory 2Gi --cpu 2

# Frontend — Streamlit dashboard
gcloud run deploy helix-frontend `
  --source frontend/ `
  --region europe-west2 `
  --allow-unauthenticated
```

## 1.3 Final configuration

| Setting | Value | Reason |
|---|---|---|
| Region | `europe-west2` (London) | Lowest latency for a UK user base |
| Python runtime | 3.13 (pinned via `.python-version`) | Torch wheel availability — see Section 2 |
| Dependencies | CPU-only wheels | The service has no GPU; GPU packages waste image size and can fail to install |
| Memory / CPU | 2 GiB / 2 vCPU | Model loading at startup exceeded the default envelope |
| Scaling | Scale-to-zero | The platform costs (almost) nothing while idle |


---

# Section 2: Build and Runtime Troubleshooting

The deployment did not work first time, and the failures are reported here deliberately: each one changed the final configuration, and diagnosing them was where most of this phase's learning happened.

## 2.1 Build failures (initial deployment)

| # | Failure | Diagnosis | Fix |
|---|---|---|---|
| 1 | Backend build failed resolving dependencies | Buildpack defaulted to a Python version with **no compatible PyTorch wheels** | Pin the runtime with a `.python-version` file (3.13) |
| 2 | Image build slow, then failed | ML stack pulled **GPU (CUDA) dependencies** onto a CPU-only service | Constrain `requirements.txt` to CPU wheels |
| 3 | Container started, then crashed on first request | **Model files missing** — they sat outside the deployed `backend/` source directory | Copy all model artefacts into `backend/models/` |
| 4 | Container killed during startup | Loading four models + ontology **exceeded the default memory/CPU envelope** | Provision `--memory 2Gi --cpu 2` |
| 5 | Frontend rendered `?` for bullets | **Encoding corruption** from local editing | Re-save sources as UTF-8 |

## 2.2 Two runtime defects found by the verification suite in Section 4

The build failures above were loud: the deployment simply refused to start. The two defects below were the opposite, and they are the more instructive pair. The service returned HTTP 200 and a plausibly-formatted response every time; only comparing its output against the same model running locally revealed that one route was wrong.

**Defect A — silent zero-filling of missing features.** The input helper built its feature row with `payload.get(feature, 0)`, so any training feature a client omitted silently became zero. A request missing five of sixteen features still returned HTTP 200, but the model was being asked to predict from a point far outside its training distribution. The calorie route returned 22 kcal for a 1.3-hour session where the same model returned 920 kcal locally. The fix derives engineered features server-side from raw measurements (BMI, HRR, intensity index, lean mass, and the interaction term all follow deterministically from the Phase 2 formulas) and returns **HTTP 422 naming the missing fields** for anything that cannot be derived. Failing loudly is the correct behaviour: a wrong prediction delivered with a 200 is worse than an error.

**Defect B — XGBoost version mismatch across environments.** With the input contract fixed, the calorie route still returned 16 kcal against 920 locally. The cause was serialisation: the model was trained and pickled under **XGBoost 3.3.0** in Colab, while `requirements.txt` pinned **2.1.0** in the container. Loading a 3.x pickle under 2.1.0 raises no error and preserves the feature count and names, but the restored booster does not reproduce the trained model's predictions. Scikit-learn was pinned identically in both environments, which is exactly why the RandomForest body-fat model was unaffected and the fault looked model-specific rather than environmental. Aligning the pin to `xgboost==3.3.0` resolved it, and the deployed route now returns **920.0 kcal — identical to the local prediction**.

Both defects share a lesson worth stating plainly: a deployed model can be structurally intact, return well-formed responses, and still be wrong. Component tests passing and the container running are not evidence of correctness. The only thing that caught these was comparing live API output against the same artefact executed independently, which is why Section 4 exists in this notebook rather than a checklist saying "deployment verified".


---

# Section 3: What Was Deployed

The backend loads, at startup:

| Artefact | File | Origin |
|---|---|---|
| Body-fat model (tuned Random Forest) | `bodyfat_rf_tuned.joblib` | Phase 3 |
| Calorie model (XGBoost) | `calories_best.joblib` | Phase 3 |
| Archetype classifier + label encoder | `archetype_classifier.joblib`, `archetype_label_encoder.joblib` | Phase 3 |
| Recovery LSTM + scaler | `recovery_lstm.pt`, `recovery_lstm_scaler.joblib` | Phase 4 |
| Knowledge graph | `helix_ontology.ttl` (217 RDF triples) | Phase 5 |

The frontend holds no models or data of its own — it is a pure client of the backend API, which is what makes the architecture a genuinely distributed, loosely-coupled system rather than a monolith split across two URLs.


---

# Section 4: Live Verification (executable)

The cells below test the deployed platform **as it runs right now**. They require nothing but an internet connection — no Drive mount, no credentials.

**Note on cold starts:** both services scale to zero when idle. The first request after a quiet period may take several seconds while Cloud Run starts a container; the timing cell below measures this explicitly rather than hiding it.


In [1]:
# ════════════════════════════════════════════════════════════════
#  Section 4.1 — Service reachability + cold-start timing
# ════════════════════════════════════════════════════════════════
import requests, time

BACKEND  = 'https://helix-backend-150551383910.europe-west2.run.app'
FRONTEND = 'https://helix-frontend-150551383910.europe-west2.run.app'

for name, url in [('Backend', BACKEND), ('Frontend', FRONTEND)]:
    t0 = time.time()
    r = requests.get(url, timeout=120)
    dt = time.time() - t0
    print(f"{name:9s} {url}")
    print(f"          HTTP {r.status_code}   response time {dt:5.2f}s"
          f"   {'(likely cold start)' if dt > 3 else '(warm)'}")


Backend   https://helix-backend-150551383910.europe-west2.run.app
          HTTP 200   response time  1.60s   (warm)
Frontend  https://helix-frontend-150551383910.europe-west2.run.app
          HTTP 200   response time  5.33s   (likely cold start)


In [2]:
# ════════════════════════════════════════════════════════════════
#  Section 4.2 — Discover the backend API surface from its own schema
# ════════════════════════════════════════════════════════════════
# FastAPI publishes an OpenAPI schema; reading it from the live service
# documents the deployed API without hard-coding assumptions about it.

schema = requests.get(f"{BACKEND}/openapi.json", timeout=60).json()
print(f"API title:   {schema['info'].get('title')}")
print(f"API version: {schema['info'].get('version')}")
print(f"\nLive endpoints ({len(schema['paths'])}):")
for path, methods in schema['paths'].items():
    for m in methods:
        print(f"   {m.upper():6s} {path}")


API title:   HELIX API
API version: 1.0

Live endpoints (7):
   GET    /
   POST   /predict/bodyfat
   POST   /predict/calories
   POST   /predict/archetype
   GET    /lstm/users
   GET    /predict/recovery/{user_id}
   GET    /ontology/philosophy/{name}


In [3]:
# ════════════════════════════════════════════════════════════════
#  Section 4.3 — Exercise every GET endpoint (with real path params)
# ════════════════════════════════════════════════════════════════
# The recovery route needs a real held-out user id, so it is fetched
# from the live API rather than hard-coded.

users = requests.get(f'{BACKEND}/lstm/users', timeout=60).json()['users']
test_user = users[0]

get_checks = ['/', '/lstm/users', f'/predict/recovery/{test_user}',
              '/ontology/philosophy/HIT', '/ontology/philosophy/HighVolume']

print(f"{'Endpoint':42s}  Status  Preview")
print('-' * 90)
for path in get_checks:
    r = requests.get(BACKEND + path, timeout=60)
    body = r.text.replace('\n', ' ')[:70]
    flag = '✅' if r.status_code == 200 else '❌'
    print(f'{flag} GET {path:36s}  {r.status_code:3d}   {body}')


Endpoint                                    Status  Preview
------------------------------------------------------------------------------------------
✅ GET /                                     200   {"status":"HELIX API running","models":["bodyfat","calories","archetyp
✅ GET /lstm/users                           200   {"users":[4020332650,4319703577,4702921684,5577150313,6391747486,69621
✅ GET /predict/recovery/4020332650          200   {"user_id":4020332650,"dates":["2016-03-19","2016-03-20","2016-03-21",
✅ GET /ontology/philosophy/HIT              200   {"philosophy":"HIT","principles":["Workouts should be brief; excessive
✅ GET /ontology/philosophy/HighVolume       200   {"philosophy":"HighVolume","principles":["Resistance must increase pro


## 4.4 End-to-end verification: all four models and the knowledge graph

The final check exercises every component of the platform in one pass: the three predictive endpoints, the recovery LSTM on a genuinely held-out user, and SPARQL-backed retrieval from the ontology.

The payload carries **raw measurements plus categorical choices**; BMI, HRR, the training-intensity index, lean body mass, and the frequency-by-experience interaction are all derived server-side (Defect A, Section 2.2). Anything the server cannot derive and the client has not supplied produces a 422 naming the missing fields rather than a silently zero-filled prediction.


In [4]:
# ════════════════════════════════════════════════════════════════
#  Section 4.4 — End-to-end prediction + recovery + ontology query
# ════════════════════════════════════════════════════════════════

features = {
    # raw measurements — engineered features derived server-side
    'Age': 28, 'Weight_kg': 82.0, 'Height_m': 1.78,
    'Max_BPM': 185, 'Avg_BPM': 128, 'Resting_BPM': 62,
    'Session_Duration_hours': 1.3, 'Calories_Burned': 900.0,
    'Fat_Percentage': 24.0, 'Water_Intake_liters': 2.5,
    'Workout_Frequency_days_per_week': 4, 'Experience_Level': 2,
    # categorical choices the server cannot infer
    'Gender_Male': 1,
    'Workout_Type_HIIT': 1, 'Workout_Type_Strength': 0, 'Workout_Type_Yoga': 0,
    'Age_Group_26-35': 1, 'Age_Group_36-45': 0, 'Age_Group_46-55': 0, 'Age_Group_56+': 0,
    'BMI_Category_Obese': 0, 'BMI_Category_Overweight': 0, 'BMI_Category_Underweight': 0,
    'Training_Commitment_Dedicated': 1, 'Training_Commitment_Regular': 0,
}

print('=' * 70)
print(' PREDICTION ENDPOINTS')
print('=' * 70)
for route in ['bodyfat', 'calories', 'archetype']:
    r = requests.post(f'{BACKEND}/predict/{route}', json={'features': features}, timeout=60)
    print(f'POST /predict/{route:10s}  HTTP {r.status_code}  →  {r.json()}')

print()
print('=' * 70)
print(' RECOVERY LSTM (held-out test user)')
print('=' * 70)
users = requests.get(f'{BACKEND}/lstm/users', timeout=60).json()['users']
uid = users[0]
rec = requests.get(f'{BACKEND}/predict/recovery/{uid}', timeout=60).json()
print(f'User {uid}: {len(rec["predicted"])} daily forecasts')
print(f'  Last 3 predicted readiness: {rec["predicted"][-3:]}')
print(f'  Last 3 actual readiness:    {rec["actual"][-3:]}')

print()
print('=' * 70)
print(' KNOWLEDGE GRAPH (SPARQL-backed philosophy lookup)')
print('=' * 70)
for name in ['HIT', 'HighVolume']:
    data = requests.get(f'{BACKEND}/ontology/philosophy/{name}', timeout=60).json()
    print(f'\nPhilosophy: {data["philosophy"]}')
    for i, p in enumerate(data.get('principles', []), 1):
        print(f'  {i}. {p}')


 PREDICTION ENDPOINTS
POST /predict/bodyfat     HTTP 200  →  {'predicted_body_fat_pct': 27.1}
POST /predict/calories    HTTP 200  →  {'predicted_calories': 920.0}
POST /predict/archetype   HTTP 200  →  {'predicted_archetype': 'General Population'}

 RECOVERY LSTM (held-out test user)
User 4020332650: 55 daily forecasts
  Last 3 predicted readiness: [49.0, 44.5, 39.8]
  Last 3 actual readiness:    [22.2, 40.5, 31.7]

 KNOWLEDGE GRAPH (SPARQL-backed philosophy lookup)

Philosophy: HIT
  1. Workouts should be brief; excessive volume is counterproductive to growth.
  2. Training must be infrequent to allow full recovery between sessions.
  3. Training must reach maximal intensity — taking each set to momentary muscular failure.
  4. Training should follow rational, scientific principles rather than tradition or volume for its own sake.
  5. Resistance must increase progressively over time to drive continued adaptation.
  6. Recovery, not training, is the true limiting factor in muscle grow

### Interpreting the verification output

**Predictions.** The calorie route returns 920.0 kcal, matching the prediction obtained by loading `calories_best.joblib` directly in Colab for the same trainee — the deployed model now reproduces the trained model exactly. This is a plumbing result rather than a modelling one: it confirms the serving path is faithful, while the modelling evidence remains the held-out R² reported in Phase 3.

**Recovery LSTM.** The forecasts for user 4020332650 track the direction and rough level of the actual readiness series (49.0/44.5/39.8 predicted against 22.2/40.5/31.7 actual) without matching it point for point, which is what R² = 0.77 looks like day by day on a user the model never saw during training.

**Knowledge graph.** Both philosophies return their principles through the SPARQL query embedded in the endpoint, confirming the ontology is loaded and queryable in the deployed environment and not merely in the Phase 5 notebook. HIT returns six principles against HighVolume's one, which reflects the modelling decision taken in Phase 5: HIT was modelled in depth as a demonstration of representational adequacy, while the other philosophies carry lighter A-Box coverage.


---

# Section 5: Phase 6 Complete

This phase closed RQ6. Across five sections we have:

- Documented the buildpack-based, Dockerfile-free deployment method and its full configuration
- Reported the five build failures and the **two silent runtime defects** — zero-filled features and an XGBoost version mismatch — with the diagnosis and fix for each
- Inventoried exactly which artefacts from Phases 3–5 the live backend serves
- Verified the deployment end-to-end: reachability and cold-start behaviour, the API surface read from the service's own schema, every GET route answering with real parameters, all three predictive endpoints returning predictions that match the trained models, the recovery LSTM forecasting for a held-out user, and SPARQL retrieval from the live ontology

Every component built in this project — statistical findings, predictive models, the recovery LSTM, and the knowledge graph — is served through one public platform:

- **Frontend:** https://helix-frontend-150551383910.europe-west2.run.app
- **Backend:** https://helix-backend-150551383910.europe-west2.run.app

The wider lesson from this phase is that deployment is not the last step of a project but a distinct source of failure modes. Four models that were correct in Phase 3 and Phase 4 could still be served incorrectly, and no amount of offline validation would have revealed it. Only executing the live system and comparing its output against the original artefacts did.

HELIX is a research prototype and decision-support tool, not a medical device.
